# Phase 5b — Benchmark a single-cell foundation model (scGPT) vs scVI

**Cross-species human/mouse islet integration — foundation-model stretch.**

> ⚠️ **Not executed in the core build.** This needs a GPU, so it runs on **Google Colab**
> (Runtime → Change runtime type → GPU), not the laptop pipeline. No numbers are pre-filled —
> you run it and read the real result. Library APIs move fast; if a call fails, check the
> installed version's docstring before assuming this code is wrong (same rule as CLAUDE.md).

**What this does:** loads the *pretrained* scGPT model, makes a zero-shot embedding of the same
human+mouse ortholog cells, and scores human→mouse label transfer with the identical kNN harness
used for scVI — apples-to-apples. Question: does a model pretrained on tens of millions of cells
beat a scVI model trained only on these ~10k cells?

In [ ]:
# 1. Environment (Colab GPU). scGPT pins are finicky; this combo is known to work.
!pip -q install scgpt==0.2.1 scanpy anndata torchtext==0.14.1 numba scikit-learn gdown
import torch; print("CUDA:", torch.cuda.is_available())  # must print True

## 2. Upload your joint data
Upload `data/processed/joint_scvi.h5ad` from the repo (already carries `X_scVI`, cell types, species).

In [ ]:
from google.colab import files
files.upload()   # choose joint_scvi.h5ad
import anndata as ad
adata = ad.read_h5ad("joint_scvi.h5ad"); print(adata)

## 3. Download the pretrained scGPT checkpoint
The `scGPT_human` whole-body checkpoint. If the Drive ID changes, get the current link from
https://github.com/bowang-lab/scGPT (README → Pretrained models).

In [ ]:
import gdown, os, pathlib
ckpt = pathlib.Path("scGPT_human"); ckpt.mkdir(exist_ok=True)
gdown.download_folder("https://drive.google.com/drive/folders/1oWh_-ZRdhtoGQ2Fw24HP41FgLoomVo-y",
                      output=str(ckpt), quiet=False, use_cookies=False)
print(os.listdir(ckpt))  # expect args.json, best_model.pt, vocab.json

## 4. Zero-shot cell embedding with scGPT
`embed_data` tokenizes each cell into scGPT's gene vocabulary and returns a per-cell embedding.
Genes outside the vocab are dropped — expect a coverage note.

In [ ]:
import scgpt as scg
adata.var["gene_name"] = adata.var_names   # our var_names are human symbols (mouse already mapped)
embed = scg.tasks.embed_data(adata, model_dir="scGPT_human",
                             gene_col="gene_name", batch_size=64, return_new_adata=True)
adata.obsm["X_scGPT"] = embed.X
print("scGPT embedding:", adata.obsm["X_scGPT"].shape)

## 5. Benchmark: human → mouse label transfer (same harness as scVI)
Identical kNN protocol + label-shuffle negative control. Compare `X_scGPT` (zero-shot foundation
model) vs `X_scVI` (trained on these cells).

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score

def label_transfer(adata, rep, shuffle=False, seed=0):
    is_h = (adata.obs["species"] == "human").values
    y = adata.obs["cell_type"].astype(str).values
    ytr = y[is_h].copy()
    if shuffle: ytr = np.random.default_rng(seed).permutation(ytr)
    clf = KNeighborsClassifier(n_neighbors=15).fit(adata.obsm[rep][is_h], ytr)
    pred = clf.predict(adata.obsm[rep][~is_h])
    return accuracy_score(y[~is_h], pred), balanced_accuracy_score(y[~is_h], pred)

for rep in ["X_scVI", "X_scGPT"]:
    a,b = label_transfer(adata, rep); a_s,b_s = label_transfer(adata, rep, shuffle=True)
    print(f"{rep:8s}  acc={a:.3f} bal_acc={b:.3f}  | shuffled acc={a_s:.3f} bal_acc={b_s:.3f}")
print("\nReference (laptop pipeline): scVI bal_acc=0.860, contrastive(sup)=0.919, PCA=0.632")

## 6. (Optional) Fine-tune scGPT for cell-type classification
Zero-shot is the fair first benchmark. For the full "fine-tune a foundation model" checkmark,
follow scGPT's annotation tutorial (repo → `tutorials/Tutorial_Annotation.ipynb`): fine-tune on
human labels, predict mouse, re-run the harness. **Report whatever you get — do not assume it
beats scVI.** On a small clean dataset a foundation model often does *not* win, and saying so is
a stronger answer than a number you can't defend.

**Talking points:** scGPT ≥ scVI → pretraining priors transferred (argue for foundation models at
scale). scGPT < scVI → a model trained directly on the 10k target cells beat a general one; the
finding is that pretraining scale doesn't automatically help a narrow, clean task. Both are honest,
defensible conclusions and both use the JD's own "benchmark and refine" framing.